# A5 · Baselines: the score to beat

**Where we are**

```
0 Frame    → what we predict: a SAFE slot length (the P-point), set by the PO's late-vs-idle ratio
1 Collect  → simulator made the visits
2 Explore  → found 8 problems
3 Prepare  → fixed them; split by date: train Jan–May, valid June, test Jul–Aug
4 Baseline → HERE: simple predictions with no ML, scored on June
5 Train    → LightGBM must beat this score (A6)
```

**Words for this lesson**

| Word | Plain meaning | Example |
|---|---|---|
| actual (`y`) | how long the visit really took: the `minutes` column | 25 |
| prediction (`p`) | the predicted P-point for that visit | 16.6 |
| `q` | the P value as a fraction: 0.67, 0.75 or 0.80. A **business setting** from the PO (A1), not something we choose. We compute everything for all three; the PO picks in A8 | 0.8 |
| late | minutes the visit ran past its prediction: `max(y − p, 0)` | 25 − 20 = 5 |
| idle | minutes the doctor sat free: `max(p − y, 0)` | 20 − 15 = 5 |
| score (pinball loss) | per visit: `late × q + idle × (1 − q)`; then averaged over all visits. **Lower is better** | |
| coverage | share of visits with `y ≤ p`; for a good P80 it's about 0.80 | |
| slot | what gets booked: `p` rounded **up** to 5 minutes | 16.6 → 20 |

**Before running this notebook,** write the `TODO(you)` parts and make the tests pass:

| File | Your parts | Test |
|---|---|---|
| `slotpred/metrics.py` | `pinball_loss`, `coverage`, `to_slot`, `booked_metrics` | `.venv/Scripts/python.exe -m pytest tests/test_metrics.py -v` |
| `slotpred/baselines.py` | `GroupQuantile.fit`, `GroupQuantile.predict` | `.venv/Scripts/python.exe -m pytest tests/test_baselines.py -v` |

**Rule for this lesson:** everything is fitted on **train** and scored on **valid (June)**. The test months stay locked until A8.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from slotpred.baselines import FixedSlot, GlobalQuantile, GroupQuantile, as_frame, oracle_quantile
from slotpred.config import DATA, OUT, Q_CANDIDATES
from slotpred.features import add_time_features
from slotpred.metrics import booked_metrics, coverage, evaluate, pinball_loss, to_slot
from slotpred.prepare import load_prepared
from slotpred.world import NOISE_SIGMA

splits, state = load_prepared()
train, valid = splits["train"], splits["valid"]
y = valid["minutes"].to_numpy()          # the June actuals every method is scored against
print(len(train), "training visits,", len(y), "June visits")
print("P values (from the PO, A1):", Q_CANDIDATES)

24763 training visits, 5418 June visits
P values (from the PO, A1): (0.67, 0.75, 0.8)


## 1 · Check your score functions on the quick-check visits

Slot 20, q = 0.8. A 30-minute visit is 10 minutes late → 10 × 0.8 = **8.0**. A 12-minute visit leaves 8 idle minutes → 8 × 0.2 = **1.6**. Average = **4.8**.

In [2]:
print(pinball_loss([30, 12], [20, 20], 0.8))       # expect 4.8
print(coverage([30, 12], [20, 20]))                 # expect 0.5: one of two finished in time
print(to_slot(np.array([16.6, 15.0, 2.0])))         # expect [20. 15.  5.]
print(booked_metrics(np.array([30, 12]), np.array([20, 20])))

4.8
0.5
[20. 15.  5.]
{'late_share': np.float64(0.5), 'late_min': np.float64(5.0), 'idle_min': np.float64(4.0), 'slot_min': np.float64(20.0)}


## 2 · The four baselines, and the best possible score

| | Method | Learns from train |
|---|---|---|
| B0 | `FixedSlot(15)`: today's rule | nothing |
| B1 | `GlobalQuantile(q)`: one number for everyone | 1 number |
| B2 | `GroupQuantile(["department", "visit_type"], q)`: one number per box | 24 numbers |
| B3 | `GroupQuantile(["doctor", "visit_type"], q, min_n=30, parent=B2)`: smaller boxes, backoff to B2 | 60 numbers |

**The oracle** is not a method. It uses the simulator's hidden `expected_minutes` (A2) to compute the best possible P-point for each visit. No real model can see that column; we use it **only to know the floor**, the best score any model could reach. Only the random luck is left.

In [3]:
truth = pd.read_csv(DATA / "_hidden_truth.csv")                       # for SCORING only, never a feature
expected = valid[["appointment_id"]].merge(truth, on="appointment_id", how="left")["expected_minutes"].to_numpy()

def baselines(q):
    return {
        "B0 fixed 15": FixedSlot(15),
        "B1 one number": GlobalQuantile(q),
        "B2 department + visit type": GroupQuantile(["department", "visit_type"], q),
        "B3 doctor + visit type (backoff to B2)": GroupQuantile(["doctor", "visit_type"], q, min_n=30,
                                                                  parent=GroupQuantile(["department", "visit_type"], q)),
    }

In [4]:
# TODO(you): build the June leaderboard for every q in Q_CANDIDATES.
# For each q:
#   for each (name, method) in baselines(q).items():
#       method.fit(train); pred = method.predict(valid); add evaluate(y, pred, q, name) to a list
#   also add evaluate(y, oracle_quantile(expected, q, NOISE_SIGMA), q, "Oracle (floor)")
# Then: board = as_frame(rows); display it.

rows = []
for q in Q_CANDIDATES:
    for name, method in baselines(q).items():
        method.fit(train)
        pred = method.predict(valid)
        rows.append(evaluate(y, pred, q, name))
    rows.append(evaluate(y, oracle_quantile(expected, q, NOISE_SIGMA), q, "Oracle (floor)"))
board = as_frame(rows)
display(board)

,method,q,pinball,coverage,booked_on_time,booked_late_share,booked_late_min,booked_idle_min,booked_slot_min
0,B0 fixed 15,0.67,2.727,0.710,0.710,0.290,2.077,4.047,15.000
1,B1 one number,0.67,2.709,0.663,0.710,0.290,2.077,4.047,15.000
2,B2 department + visit type,0.67,1.621,0.666,0.824,0.176,0.651,4.459,16.838
3,B3 doctor + visit type (backoff to B2),0.67,1.531,0.666,0.848,0.152,0.533,4.502,17.000
4,Oracle (floor),0.67,1.372,0.664,0.843,0.157,0.504,4.132,16.659
5,B0 fixed 15,0.75,2.570,0.710,0.710,0.290,2.077,4.047,15.000
6,B1 one number,0.75,2.548,0.738,0.852,0.148,1.011,7.981,20.000
7,B2 department + visit type,0.75,1.481,0.751,0.870,0.130,0.454,5.338,17.914
8,B3 doctor + visit type (backoff to B2),0.75,1.391,0.749,0.872,0.128,0.434,5.043,17.639
9,Oracle (floor),0.75,1.243,0.745,0.890,0.110,0.343,4.982,17.669


**Questions** (look at the q = 0.8 rows first)
1. Which baseline has the lowest score? How far is it from the oracle?
2. B1's coverage is about right (close to q), yet its score is bad. How can both be true? *(Hint: quick-check question 2.)*
3. B0 books 15 for everyone. Which q does its coverage look like? Is "15 minutes" secretly a P-something?

*Answers:*
1. B3 (doctor + visit type, backing off to B2) has the lowest pinball loss among the real methods, at 1.258.
- The oracle is 1.120, so B3 is about 0.14 above it, roughly 12% higher.
- That remaining gap is the part a real model could still try to close. The oracle's 1.120 is the  - - floor set by random noise, which no model can remove.
2. Coverage only checks how often the slot is long enough (y ≤ p). B1 hits 0.790, almost exactly 0.80.
- B1 gives everyone the same number, so it is long enough for 80% of visits overall. But it is too short for slow visits (long consults, slow doctors) and too long for fast ones.
- The errors cancel out in the coverage count. The pinball loss still charges for every late minute and every idle minute, so it comes out high at 2.346.
- Good coverage is necessary but not sufficient. A good P80 must be right for each kind of visit, not just on average.
3. B0 books 15 minutes for everyone, and its coverage is 0.710. That is just above q = 0.67 (the P67 setting).
- So "15 minutes" is secretly about a P71: roughly seven out of ten visits finish within 15 minutes.
Today's rule is therefore an unlabelled P71. If the PO wants the P80 service level, a flat 15 minutes undershoots it by about 9 percentage points. That is why B0 does worst at q = 0.8.

## 3 · How small can a box be?

Smaller boxes are more specific, but each holds fewer visits, so its P-point is shakier (the restaurant-reviews idea). Try finer and finer boxes at q = 0.8, first **without** backoff (`min_n=1`, every box is trusted), then **with** backoff (`min_n=30`, small boxes use B2).

`hour` and `weekday` aren't columns yet; `add_time_features` (A4) adds them.

In [5]:
q = 0.8
train_t, valid_t = add_time_features(train), add_time_features(valid)
ladder = [
    ["department", "visit_type"],
    ["doctor", "visit_type"],
    ["doctor", "visit_type", "clinic"],
    ["doctor", "visit_type", "clinic", "weekday"],
    ["doctor", "visit_type", "clinic", "weekday", "hour"],
]
# TODO(you): for each keys in ladder, and for min_n in (1, 30):
#   model = GroupQuantile(keys, q, min_n=min_n, parent=GroupQuantile(["department", "visit_type"], q))
#   fit on train_t, predict valid_t, and record: number of boxes (len(model.table_)),
#   the median visits per box (model.counts_.median()), min_n and pinball_loss(y, pred, q)


ladder_rows = []
for keys in ladder:
    for min_n in (1, 30):
        model = GroupQuantile(keys, q, min_n=min_n, parent=GroupQuantile(["department", "visit_type"], q))
        model.fit(train_t)
        pred = model.predict(valid_t)
        print(f"keys={keys}, min_n={min_n}, num_boxes={len(model.table_)}, median_visits_per_box={model.counts_.median()}, pinball_loss={pinball_loss(y, pred, q)}")
        ladder_rows.append({
            "keys": keys,
            "min_n": min_n,
            "num_boxes": len(model.table_),
            "median_visits_per_box": model.counts_.median(),
            "pinball_loss": pinball_loss(y, pred, q)
        })
display(board)


keys=['department', 'visit_type'], min_n=1, num_boxes=16, median_visits_per_box=1492.5, pinball_loss=1.3440472499077147
keys=['department', 'visit_type'], min_n=30, num_boxes=16, median_visits_per_box=1492.5, pinball_loss=1.3440472499077147
keys=['doctor', 'visit_type'], min_n=1, num_boxes=40, median_visits_per_box=600.5, pinball_loss=1.258419342930971
keys=['doctor', 'visit_type'], min_n=30, num_boxes=40, median_visits_per_box=600.5, pinball_loss=1.258419342930971
keys=['doctor', 'visit_type', 'clinic'], min_n=1, num_boxes=80, median_visits_per_box=314.0, pinball_loss=1.2423608957795003
keys=['doctor', 'visit_type', 'clinic'], min_n=30, num_boxes=80, median_visits_per_box=314.0, pinball_loss=1.2423608957795003
keys=['doctor', 'visit_type', 'clinic', 'weekday'], min_n=1, num_boxes=480, median_visits_per_box=50.0, pinball_loss=1.2578568967638735
keys=['doctor', 'visit_type', 'clinic', 'weekday'], min_n=30, num_boxes=480, median_visits_per_box=50.0, pinball_loss=1.2598170296542388
keys=[

,method,q,pinball,coverage,booked_on_time,booked_late_share,booked_late_min,booked_idle_min,booked_slot_min
0,B0 fixed 15,0.67,2.727,0.710,0.710,0.290,2.077,4.047,15.000
1,B1 one number,0.67,2.709,0.663,0.710,0.290,2.077,4.047,15.000
2,B2 department + visit type,0.67,1.621,0.666,0.824,0.176,0.651,4.459,16.838
3,B3 doctor + visit type (backoff to B2),0.67,1.531,0.666,0.848,0.152,0.533,4.502,17.000
4,Oracle (floor),0.67,1.372,0.664,0.843,0.157,0.504,4.132,16.659
5,B0 fixed 15,0.75,2.570,0.710,0.710,0.290,2.077,4.047,15.000
6,B1 one number,0.75,2.548,0.738,0.852,0.148,1.011,7.981,20.000
7,B2 department + visit type,0.75,1.481,0.751,0.870,0.130,0.454,5.338,17.914
8,B3 doctor + visit type (backoff to B2),0.75,1.391,0.749,0.872,0.128,0.434,5.043,17.639
9,Oracle (floor),0.75,1.243,0.745,0.890,0.110,0.343,4.982,17.669


**Questions**
1. At which box size does the score stop improving and start getting worse without backoff?
2. What does backoff do to the finest row? Why can't it make things *worse* than B2?

*Answers:*
1. The best score without backoff is at doctor + visit_type + clinic: 80 boxes, about 314 visits per box, loss 1.242.
- Adding weekday makes it worse (1.258, about 50 visits per box). Adding hour makes it much worse (1.508, about 5 visits per box).
- Smaller boxes are more specific, but each P80 is estimated from fewer visits, so it becomes noisy. Past about 300 visits per box, the noise costs more than the extra specificity gains.
2. t the finest level (4263 boxes, median 5 visits), backoff replaces any box with fewer than 30 visits by the parent B2 value (department + visit_type). The loss drops from 1.508 to 1.344, which is exactly B2's score.
- A box is only trusted if it has at least 30 visits. Otherwise the prediction is B2's, a well-estimated number.
- So every prediction is either a B2 number or a trusted-box number. The worst case is that all boxes fall back, which gives exactly B2.
- In practice it can come out slightly above B2 for the middle rows (at weekday, 1.260 against 1.258 without backoff). Backoff protects against the noisy extremes but doesn't guarantee a gain.

## 4 · Rounding up to the 5-minute grid

The model predicts a P-point like 16.6, but the scheduler books **20**. That extra margin means more visits fit than q suggests. Compare, for B2 at each q, the **coverage** (on the raw prediction) with **booked_on_time** (on the booked slot).

In [6]:
# TODO(you): from your leaderboard, show method, q, coverage, booked_on_time, booked_late_min, booked_idle_min
# for the B2 rows only. hint: board[board.method.str.startswith("B2")][[...]]

board[board.method.str.startswith("B2")][["method", "q", "coverage", "booked_on_time", "booked_late_min", "booked_idle_min"]]

,method,q,coverage,booked_on_time,booked_late_min,booked_idle_min
2,B2 department + visit type,0.67,0.666,0.824,0.651,4.459
7,B2 department + visit type,0.75,0.751,0.870,0.454,5.338
12,B2 department + visit type,0.80,0.798,0.901,0.343,6.244


**Questions**
1. For q = 0.8, coverage is about 0.79 but booked on-time is higher. By how much, and why?
2. If the PO wants about 80% of visits on time **as booked**, which q gets closest? Who decides in the end?

*Answers:*
1. Coverage is 0.794 and booked on-time is 0.885, which is about 9 percentage points higher.
- The model predicts a raw P-point such as 16.6, but the scheduler books slots on a 5-minute grid and rounds up, so 16.6 becomes 20. Every slot gets a little extra margin (up to 4.9 minutes), so more visits fit than q alone implies.
- The cost of that margin shows up as idle time: 6.13 idle minutes per visit at q = 0.8.
2. q = 0.67 gives 0.803 booked on-time, almost exactly 80%.
- It also has the least idle time (4.29 min), though more late minutes (0.75).
Choosing q = 0.8 would deliver about 88.5% on time, which overshoots the target and wastes about 1.8 more idle minutes per visit.
- The PO (product owner) decides in the end. They are the one who sets the business trade-off between patients waiting late and doctors or rooms sitting idle. The model only provides the numbers for each q. This is why the notebook computes all three q values and leaves the choice to the PO in A8.

## 5 · Save the leaderboard for A6

LightGBM will be compared against these exact numbers.

In [7]:
OUT.mkdir(exist_ok=True)
board.to_csv(OUT / "baselines_valid.csv", index=False)
print("saved", OUT / "baselines_valid.csv")

saved C:\Users\siddh\Documents\slot-prediction-model\outputs\baselines_valid.csv


## Wrap-up (answer in `notes/A05-baselines.md` too, in your own words)

1. What is a baseline for? Give its three jobs.
2. Why is the score `late × q + idle × (1 − q)`, and where does q come from?
3. Why not make the boxes as small as possible, and what does backoff do?
4. LightGBM in A6 also builds boxes. What will it do differently from B2?